# What Actually Drives Streaming Performance?
**Multiple Linear Regression · K-Means Clustering · PCA · 952 Songs**

Playlist placement, audio features, release timing — which actually predicts whether a song performs? This project analyzed Spotify's top songs of 2023 to find out, testing variables across Spotify, Apple Music, and Deezer.

**Research questions:**
- Which platform's playlist placements have the strongest effect on streams?
- Do audio features (danceability, energy, valence, tempo) predict streaming success?
- Does release season affect which performance tier a song lands in?

## 1. Setup & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from IPython.display import Image, display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import ttest_ind, chi2_contingency

warnings.filterwarnings('ignore')

DATA_DIR  = '/content/drive/MyDrive/spotify'
PLOTS_DIR = 'plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

for old_file in ['streams_distribution.png','playlist_impact_comparison.png',
    'actual_vs_predicted_streams.png','seasonal_performance_distribution.png',
    'playlist_clustering.png','pca_clusters.png',
    'stream_correlations.png','artist_tier_audio_features.png']:
    p = os.path.join(PLOTS_DIR, old_file)
    if os.path.exists(p): os.remove(p)

print('Setup complete.')

## 2. Visual Style & Helpers

In [ ]:
LIME  = '#c8ff00'
PINK  = '#ff69b4'
WHITE = '#ffffff'
BG    = '#0a0a0a'
GRAY  = '#333333'
MGRAY = '#1a1a1a'

GREEN_PALETTE  = ['#2a4a00', '#4d8c00', '#96cc00', '#c8ff00']
CLUSTER_PALETTE = {
    'Low Visibility':   '#2a4a00',
    'Breaking Through': '#4d8c00',
    'Well Known':       '#96cc00',
    'Phenomenal':       '#c8ff00'
}

plt.rcParams.update({
    'figure.figsize':   (10, 6),
    'figure.facecolor': BG,
    'axes.facecolor':   BG,
    'axes.edgecolor':   GRAY,
    'axes.labelcolor':  WHITE,
    'xtick.color':      WHITE,
    'ytick.color':      WHITE,
    'text.color':       WHITE,
    'axes.titleweight': 'bold',
    'axes.titlesize':   14,
    'axes.labelsize':   11,
})

def apply_style(ax):
    ax.set_facecolor(BG)
    ax.grid(True, axis='y', color=MGRAY, linewidth=0.8)
    ax.grid(False, axis='x')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color(GRAY)
    ax.spines['bottom'].set_color(GRAY)

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, filename), dpi=300, bbox_inches='tight', facecolor=BG)
    plt.show()
    plt.close()

def add_caption(fig, text):
    fig.text(0.13, 0.01, text, fontsize=8, color=GRAY, wrap=True)

def detect_csv(data_dir):
    csvs = [f for f in os.listdir(data_dir) if f.lower().endswith('.csv')]
    if not csvs: raise FileNotFoundError(f'No CSV in {data_dir}')
    if len(csvs) == 1: return os.path.join(data_dir, csvs[0])
    for p in ['spotify','songs','2023','top']:
        for fn in csvs:
            if p in fn.lower(): return os.path.join(data_dir, fn)
    return os.path.join(data_dir, csvs[0])

def clean_cols(df):
    df = df.copy()
    df.columns = [re.sub(r'[^a-z0-9_]+','_',c.strip().lower()).strip('_') for c in df.columns]
    return df

def to_num(s):
    return pd.to_numeric(s.astype(str).str.replace(',','',regex=False), errors='coerce')

def month_to_season(m):
    if m in [12,1,2]: return 'Winter'
    if m in [3,4,5]:  return 'Spring'
    if m in [6,7,8]:  return 'Summer'
    return 'Fall'

print('Style and helpers ready.')

## 3. Load Dataset

> **Data:** Top Spotify Songs 2023 — https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023
> Place the CSV in a folder called `spotify` in your Google Drive.

In [ ]:
csv_path    = detect_csv(DATA_DIR)
print('Using file:', csv_path)
spotify_raw = pd.read_csv(csv_path, encoding='latin1')
spotify_raw = clean_cols(spotify_raw)
print('\nColumns found:')
print(spotify_raw.columns.tolist())

## 4. Prepare Dataset

In [ ]:
column_map = {
    'track_name':           ['track_name'],
    'artist_name':          ['artist_s_name','artist_name','artists_name'],
    'streams':              ['streams'],
    'released_month':       ['released_month'],
    'in_spotify_playlists': ['in_spotify_playlists'],
    'in_apple_playlists':   ['in_apple_playlists'],
    'in_deezer_playlists':  ['in_deezer_playlists'],
    'danceability_pct':     ['danceability','danceability_','danceability_pct'],
    'energy_pct':           ['energy','energy_','energy_pct'],
    'valence_pct':          ['valence','valence_','valence_pct'],
    'bpm':                  ['bpm']
}
resolved = {}
for target, candidates in column_map.items():
    for c in candidates:
        if c in spotify_raw.columns:
            resolved[target] = c
            break

spotify = pd.DataFrame()
spotify['track_name']           = spotify_raw[resolved['track_name']]
spotify['streams']              = to_num(spotify_raw[resolved['streams']])
spotify['released_month']       = to_num(spotify_raw[resolved['released_month']])
spotify['in_spotify_playlists'] = to_num(spotify_raw[resolved['in_spotify_playlists']])
spotify['in_apple_playlists']   = to_num(spotify_raw[resolved['in_apple_playlists']])
spotify['in_deezer_playlists']  = to_num(spotify_raw[resolved['in_deezer_playlists']])
for col in ['danceability_pct','energy_pct','valence_pct','bpm']:
    if col in resolved: spotify[col] = to_num(spotify_raw[resolved[col]])
if 'artist_name' in resolved:
    spotify['artist_name'] = spotify_raw[resolved['artist_name']].astype(str)

spotify = spotify.dropna(subset=['streams','released_month',
    'in_spotify_playlists','in_apple_playlists','in_deezer_playlists']).copy()
spotify = spotify[spotify['streams'] > 0].reset_index(drop=True).copy()

spotify['season']           = spotify['released_month'].astype(int).apply(month_to_season)
spotify['log_streams']      = np.log1p(spotify['streams'])
spotify['performance_tier'] = pd.qcut(spotify['streams'], q=4, labels=['Lower','Mid-Lower','Mid-Upper','Top'])

if 'artist_name' in spotify.columns:
    artist_streams = spotify.groupby('artist_name')['streams'].sum()
    top_artists    = artist_streams[artist_streams >= artist_streams.quantile(0.75)].index
    spotify['is_top_artist'] = spotify['artist_name'].isin(top_artists)
else:
    spotify['is_top_artist'] = spotify['streams'] >= spotify['streams'].quantile(0.75)

print('=' * 60)
print('DATASET SUMMARY')
print('=' * 60)
print(f'Songs: {len(spotify):,}')
print(spotify[['streams','in_spotify_playlists','in_apple_playlists','in_deezer_playlists']].describe())

In [ ]:
# Streams distribution
fig, ax = plt.subplots()
sns.histplot(spotify['log_streams'], bins=40, color=LIME, edgecolor=BG, linewidth=0.3, ax=ax)
ax.set_title('Most Songs Receive Moderate Streaming Attention', pad=12)
ax.set_xlabel('Streaming Performance (scaled for readability)')
ax.set_ylabel('Number of Songs')
add_caption(fig, 'Even among the top songs of 2023, most cluster in a moderate performance range.')
apply_style(ax)
save_plot('streams_distribution.png')

## 5. Regression Model

**Q: Which platform's playlist placements most strongly predict stream counts?**

Standardized coefficients used for cross-platform comparison.

In [ ]:
model_df = spotify[['log_streams','in_spotify_playlists','in_apple_playlists','in_deezer_playlists']].dropna().copy()
X        = sm.add_constant(model_df[['in_spotify_playlists','in_apple_playlists','in_deezer_playlists']])
y        = model_df['log_streams']
model    = sm.OLS(y, X).fit()
print(model.summary())

scaled_df = model_df.copy()
for col in model_df.columns:
    scaled_df[col] = (scaled_df[col] - scaled_df[col].mean()) / scaled_df[col].std()
X_s          = sm.add_constant(scaled_df[['in_spotify_playlists','in_apple_playlists','in_deezer_playlists']])
scaled_model = sm.OLS(scaled_df['log_streams'], X_s).fit()

coef_summary = pd.DataFrame({
    'platform':    ['Spotify','Apple Music','Deezer'],
    'coefficient': [scaled_model.params['in_spotify_playlists'],
                    scaled_model.params['in_apple_playlists'],
                    scaled_model.params['in_deezer_playlists']]
}).sort_values('coefficient', ascending=False)

spotify['predicted_log_streams'] = np.nan
spotify.loc[model_df.index, 'predicted_log_streams'] = model.predict(X)
spotify['predicted_streams'] = np.expm1(spotify['predicted_log_streams'])

print(f'\nR-squared: {model.rsquared:.4f} | Adj R-squared: {model.rsquared_adj:.4f}')
print('\nStandardized coefficients:')
print(coef_summary.to_string(index=False))

In [ ]:
# Playlist platform impact
fig, ax = plt.subplots()
colors  = [LIME if c >= 0 else PINK for c in coef_summary['coefficient']]
sns.barplot(data=coef_summary, x='platform', y='coefficient',
            palette=dict(zip(coef_summary['platform'], colors)), ax=ax)
ax.axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
ax.set_title('Spotify Playlists Have the Strongest Impact on Streams', pad=12)
ax.set_xlabel('Platform')
ax.set_ylabel('Relative Influence on Streaming Performance')
apply_style(ax)
save_plot('playlist_impact_comparison.png')

# Actual vs predicted
fit_sample = spotify.dropna(subset=['predicted_streams']).sample(min(500, len(spotify)), random_state=42).copy()
fit_sample['streams_j']   = fit_sample['streams'] * np.exp(np.random.normal(0, 0.05, len(fit_sample)))
fit_sample['predicted_j'] = fit_sample['predicted_streams'] * np.exp(np.random.normal(0, 0.05, len(fit_sample)))
max_val = max(fit_sample['streams'].quantile(0.99), fit_sample['predicted_streams'].quantile(0.99))
min_val = min(fit_sample['streams'].quantile(0.01), fit_sample['predicted_streams'].quantile(0.01))

fig, ax = plt.subplots()
ax.scatter(fit_sample['streams_j'], fit_sample['predicted_j'], alpha=0.3, s=20, color=LIME)
ax.plot([min_val, max_val], [min_val, max_val], linestyle='--', color=GRAY, linewidth=1.5, label='Perfect prediction')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title(f'Model Captures Overall Streaming Trends (R²={model.rsquared:.3f})', pad=12)
ax.set_xlabel('Actual Streams')
ax.set_ylabel('Predicted Streams')
ax.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0)
apply_style(ax)
save_plot('actual_vs_predicted_streams.png')

## 6. Hypothesis Testing

In [ ]:
print('=' * 70)
print('HYPOTHESIS TESTING')
print('=' * 70)

feature_cols       = ['danceability_pct','energy_pct']
available_features = [c for c in feature_cols if c in spotify.columns]

if available_features:
    for feature in available_features:
        top_vals     = spotify[spotify['is_top_artist']][feature].dropna()
        non_top_vals = spotify[~spotify['is_top_artist']][feature].dropna()
        t_stat, p_val = ttest_ind(top_vals, non_top_vals, equal_var=False)
        print(f'\nT-test: Top vs Non-Top Artists — {feature}')
        print(f'  Top mean: {top_vals.mean():.2f} | Non-top mean: {non_top_vals.mean():.2f}')
        print(f'  t={t_stat:.4f} | p={p_val:.6f} | Significant: {"Yes" if p_val < 0.05 else "No"}')

contingency = pd.crosstab(spotify['season'], spotify['performance_tier'])
chi2_stat, chi2_p, chi2_dof, _ = chi2_contingency(contingency)
print(f'\nChi-square: Release Season vs Performance Tier')
print(f'  chi2={chi2_stat:.4f} | df={chi2_dof} | p={chi2_p:.6f} | Significant: {"Yes" if chi2_p < 0.05 else "No"}')

In [ ]:
# Seasonal performance distribution
season_order = ['Spring','Summer','Fall','Winter']
tier_order   = ['Lower','Mid-Lower','Mid-Upper','Top']
season_perf  = spotify.groupby(['season','performance_tier'], observed=False).size().reset_index(name='count')
season_perf['season']           = pd.Categorical(season_perf['season'], categories=season_order, ordered=True)
season_perf['performance_tier'] = pd.Categorical(season_perf['performance_tier'], categories=tier_order, ordered=True)
season_perf = season_perf.sort_values(['season','performance_tier'])
season_pivot = season_perf.pivot(index='season', columns='performance_tier', values='count').fillna(0)
season_pct   = season_pivot.div(season_pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots()
bottom  = np.zeros(len(season_pct))
for i, col in enumerate(season_pct.columns):
    ax.bar(season_pct.index, season_pct[col], bottom=bottom, label=col, color=GREEN_PALETTE[i])
    bottom += season_pct[col].values
ax.set_title('Song Performance Mix Changes Across Release Seasons', pad=12)
ax.set_xlabel('Release Season')
ax.set_ylabel('Share of Songs (%)')
ax.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0)
add_caption(fig, f'Winter releases show the highest share of top-tier songs (χ²={chi2_stat:.2f}, p<.001).')
apply_style(ax)
save_plot('seasonal_performance_distribution.png')

## 7. K-Means Clustering & PCA

In [ ]:
cluster_df      = spotify[['streams','in_spotify_playlists','in_apple_playlists','in_deezer_playlists']].dropna().copy()
scaled_features = StandardScaler().fit_transform(cluster_df)
cluster_df['cluster'] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(scaled_features)

cluster_medians   = cluster_df.groupby('cluster')['streams'].median().sort_values()
tier_names        = ['Low Visibility','Breaking Through','Well Known','Phenomenal']
cluster_label_map = {cid: tier_names[i] for i,(cid,_) in enumerate(cluster_medians.items())}
cluster_df['cluster_label'] = cluster_df['cluster'].map(cluster_label_map)

spotify['cluster']       = np.nan
spotify['cluster_label'] = np.nan
spotify.loc[cluster_df.index, 'cluster']       = cluster_df['cluster']
spotify.loc[cluster_df.index, 'cluster_label'] = cluster_df['cluster_label']

pca = PCA(n_components=2)
pca_result     = pca.fit_transform(scaled_features)
cluster_df['pca1'] = pca_result[:, 0]
cluster_df['pca2'] = pca_result[:, 1]

print(cluster_df['cluster_label'].value_counts())
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}')

In [ ]:
# Clustering scatter
fig, ax = plt.subplots()
sns.scatterplot(data=cluster_df, x='in_spotify_playlists', y='streams',
                hue='cluster_label', palette=CLUSTER_PALETTE, alpha=0.6, s=35, ax=ax,
                hue_order=['Low Visibility','Breaking Through','Well Known','Phenomenal'])
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Songs Group into Distinct Performance Tiers by Playlist Exposure', pad=12)
ax.set_xlabel('Number of Spotify Playlists the Song Appears In')
ax.set_ylabel('Total Streams')
ax.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0)
apply_style(ax)
save_plot('playlist_clustering.png')

# PCA visualization
fig, ax = plt.subplots()
sns.scatterplot(data=cluster_df, x='pca1', y='pca2',
                hue='cluster_label', palette=CLUSTER_PALETTE, alpha=0.7, s=35, ax=ax,
                hue_order=['Low Visibility','Breaking Through','Well Known','Phenomenal'])
ax.set_title('Four Distinct Song Performance Groups Confirmed', pad=12)
ax.set_xlabel('Overall Playlist Exposure (increases right →)')
ax.set_ylabel('Platform Mix — Apple vs Deezer Balance (increases up ↑)')
ax.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0)
add_caption(fig, f'PCA reduces 4 variables to 2D. {pca.explained_variance_ratio_.sum():.0%} variance explained.')
apply_style(ax)
save_plot('pca_clusters.png')

## 8. Audio Feature Correlations

In [ ]:
all_feature_cols = ['danceability_pct','energy_pct','valence_pct','bpm']
available_all    = [c for c in all_feature_cols if c in spotify.columns]

if available_all:
    stream_corr = spotify[available_all + ['streams']].dropna().corr()['streams'].drop('streams').sort_values()
    print('AUDIO FEATURE CORRELATIONS WITH STREAMS')
    print(stream_corr)
else:
    stream_corr = None

In [ ]:
# Top vs non-top artist audio features
if available_features:
    fc = spotify[available_features + ['is_top_artist']].dropna().copy()
    fc['artist_tier'] = fc['is_top_artist'].map({True: 'Top Artists', False: 'Other Artists'})
    fm = fc.melt(id_vars='artist_tier', value_vars=available_features, var_name='feature', value_name='score')
    fm['feature'] = fm['feature'].str.replace('_pct','').str.title()

    fig, ax = plt.subplots()
    sns.barplot(data=fm, x='feature', y='score', hue='artist_tier',
                palette={'Top Artists': LIME, 'Other Artists': PINK}, ax=ax)
    ax.set_title('Top Artists Show Slightly Higher Danceability and Energy', pad=12)
    ax.set_xlabel('Audio Feature (scored 0–100)')
    ax.set_ylabel('Average Score')
    ax.set_ylim(0, 100)
    ax.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0)
    apply_style(ax)
    save_plot('artist_tier_audio_features.png')

# Correlation bar chart
if stream_corr is not None:
    cp = stream_corr.reset_index()
    cp.columns = ['feature','correlation']
    cp['feature'] = cp['feature'].str.replace('_pct','').str.replace('_',' ').str.title().str.replace('Bpm','Tempo (BPM)')
    cp['color']   = cp['correlation'].apply(lambda x: LIME if x >= 0 else PINK)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(cp['feature'], cp['correlation'], color=cp['color'])
    ax.axvline(0, color=GRAY, linewidth=0.8, linestyle='--')
    ax.set_title('Audio Features Have Near-Zero Correlation with Streams', pad=12)
    ax.set_xlabel('Correlation with Total Streams')
    apply_style(ax)
    save_plot('stream_correlations.png')

## 9. Key Findings Summary

In [ ]:
print('=' * 70)
print('KEY FINDINGS SUMMARY')
print('=' * 70)
sp_b  = coef_summary.loc[coef_summary['platform']=='Spotify','coefficient'].values[0]
ap_b  = coef_summary.loc[coef_summary['platform']=='Apple Music','coefficient'].values[0]
dz_b  = coef_summary.loc[coef_summary['platform']=='Deezer','coefficient'].values[0]
print(f"""
REGRESSION (R² = {model.rsquared:.3f})
  Playlist placement explained {model.rsquared:.1%} of variance in streams
  Spotify:     β = {sp_b:.3f}
  Apple Music: β = {ap_b:.3f}
  Deezer:      β = {dz_b:.3f}

HYPOTHESIS TESTING
  Audio features do not predict streaming success (r ≈ 0)
  Release season significantly associated with performance tier
  (χ² = {chi2_stat:.2f}, p < .001) — winter releases peak

CLUSTERING
  Songs segment into 4 tiers by playlist exposure and streams
  PCA confirmed clusters are distinct ({pca.explained_variance_ratio_.sum():.0%} variance explained)
""")

## 10. Preview All Charts

In [ ]:
charts = [
    'streams_distribution.png',
    'playlist_impact_comparison.png',
    'actual_vs_predicted_streams.png',
    'seasonal_performance_distribution.png',
    'playlist_clustering.png',
    'pca_clusters.png',
    'artist_tier_audio_features.png',
    'stream_correlations.png'
]
print(f'Saved: {sorted(os.listdir(PLOTS_DIR))}')
for chart in charts:
    path = os.path.join(PLOTS_DIR, chart)
    if os.path.exists(path):
        print(f'--- {chart} ---')
        display(Image(filename=path))